# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Methods:** Logistic Regression and Random Forest Classifier

**Why:**
The task is binary classification ("yes/no with an observed label") with a moderately balanced base rate of ~33.2%.
* I am starting with **Logistic Regression** because simplicity is a feature; it provides interpretable coefficients to easily spot if the model is relying on spurious correlations.
* I am also training a **Random Forest Classifier** because it naturally captures non-linear interactions (e.g., a page might only need a refresh if it is *both* old AND has high impressions, which linear models handle poorly without explicit feature crossing). If Logistic Regression matches Random Forest performance, we will prefer the simpler model.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

**Design:** `GroupShuffleSplit` grouped by `client_id` (80/20 split, `random_state=42`).

**Why:**
Content from the same client shares SEO patterns, domain authority, temporal seasonality, and baseline traffic levels. If we use a standard random row-level split, pages from a high-traffic client will bleed into both the training and test sets. The model would learn to memorize client-specific traffic patterns rather than genuine content decay signatures, causing massive data leakage. Grouping by `client_id` ensures our test metrics honestly reflect how the model performs on unseen clients.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

# 1. Load Data
df = pd.read_csv('/content/content_refresh_anonymized.csv')

# 2. Build Target (Same as w02)
df['target_needs_refresh'] = (
    (df['impressions_90d'] >= 500) &
    (df['trend_direction'] == 'down')
).astype(int)

# 3. Handle Gotchas & Feature Engineering
df['ctr_decimal'] = df['ctr'] / 100.0

# avg_position: 0 means unranked. Replace 0 with NaN, then impute with max (worst) rank, and flag.
df['has_position'] = (df['avg_position'] > 0).astype(int)
df['avg_position_clean'] = df['avg_position'].replace(0, np.nan)
df['avg_position_clean'] = df['avg_position_clean'].fillna(df['avg_position_clean'].max())

# word_count: handle missingness
df['has_word_count'] = df['word_count'].notna().astype(int)
df['word_count'] = df['word_count'].fillna(0)

# Fill missing numericals with 0 for safety (e.g., GA4 metrics might be missing)
cols_to_fill = ['engagement_rate', 'scroll_rate', 'sessions_90d']
df[cols_to_fill] = df[cols_to_fill].fillna(0)

# 4. Define Safe Features (NO LEAKAGE)
safe_features = [
    'content_age_days', 'days_since_last_update', 'word_count',
    'impressions_90d', 'clicks_90d', 'ctr_decimal',
    'avg_position_clean', 'has_position', 'has_word_count',
    'engagement_rate', 'scroll_rate', 'sessions_90d'
]

X = df[safe_features]
y = df['target_needs_refresh']
groups = df['client_id']

# 5. Grouped Split
gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, y_train = X.iloc[train_idx].copy(), y.iloc[train_idx].copy()
X_test, y_test = X.iloc[test_idx].copy(), y.iloc[test_idx].copy()

# Retain original test dataframe for baseline scoring
df_test = df.iloc[test_idx].copy()

print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")
print(f"Base Rate (Train): {y_train.mean():.1%} | Base Rate (Test): {y_test.mean():.1%}")

Train size: 23837 | Test size: 6163
Base Rate (Train): 35.5% | Base Rate (Test): 24.5%


## 3. Train + compare vs my baseline

Here we evaluate the w04 rule baseline and both machine learning models on the exact same held-out test set. We prioritize **Precision@20** and **Precision@50** because a content team's weekly bandwidth is limited; false positives waste human writing hours.

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import average_precision_score

# --- 1. Reconstruct w04 Baseline on Test Set ---
# Normalize components for the baseline formula dynamically based on test set max
staleness = df_test['days_since_last_update'] / df_test['days_since_last_update'].max()
ctr_underperformance = 1.0 - df_test['ctr_decimal']
visibility = df_test['impressions_90d'] / df_test['impressions_90d'].max()

df_test['baseline_action_score'] = (0.4 * staleness) + (0.4 * ctr_underperformance) + (0.2 * visibility)

# --- 2. Train Logistic Regression ---
lr_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1.0, random_state=42, max_iter=1000)
)
lr_model.fit(X_train, y_train)
df_test['lr_prob'] = lr_model.predict_proba(X_test)[:, 1]

# --- 3. Train Random Forest ---
rf_model = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
df_test['rf_prob'] = rf_model.predict_proba(X_test)[:, 1]

# --- 4. Evaluation Helper ---
def precision_at_k(y_true, scores, k):
    top_k_indices = np.argsort(scores)[::-1][:k]
    return y_true.iloc[top_k_indices].mean()

# --- 5. Build Comparison Table ---
results = []
models = {
    'Rule Baseline (w04)': 'baseline_action_score',
    'Logistic Regression': 'lr_prob',
    'Random Forest': 'rf_prob'
}

for name, score_col in models.items():
    results.append({
        'Model': name,
        'Precision@20': f"{precision_at_k(df_test['target_needs_refresh'], df_test[score_col], 20):.1%}",
        'Precision@50': f"{precision_at_k(df_test['target_needs_refresh'], df_test[score_col], 50):.1%}",
        'PR-AUC': f"{average_precision_score(df_test['target_needs_refresh'], df_test[score_col]):.3f}"
    })

comparison_df = pd.DataFrame(results)

# Add random base rate row for context
base_rate = df_test['target_needs_refresh'].mean()
comparison_df.loc[-1] = ['Base Rate (random)', f"{base_rate:.1%}", f"{base_rate:.1%}", f"{base_rate:.3f}"]
comparison_df.index = comparison_df.index + 1
comparison_df = comparison_df.sort_index()

display(comparison_df)

,Model,Precision@20,Precision@50,PR-AUC
0,Base Rate (random),24.5%,24.5%,0.245
1,Rule Baseline (w04),20.0%,34.0%,0.242
2,Logistic Regression,40.0%,38.0%,0.258
3,Random Forest,70.0%,68.0%,0.583


## 4. Errors and interpretation

We use Permutation Importance to ensure the models aren't relying on data leakage.

**Error Analysis:**
*   **False Positives (The model cried wolf):** Pages that have high impressions and older age, making them *look* stale, but their actual trend is stable or rising (which the model can't see because `trend_direction` is hidden to prevent leakage).
*   **False Negatives (The model missed decay):** Pages with low `impressions_90d` (e.g., hovering right around the 500 threshold) that are actively bleeding traffic, but the model prioritizes pages with massive raw impression volume.

In [3]:
from sklearn.inspection import permutation_importance

# 1. Feature Importance (Sanity Check)
result = permutation_importance(rf_model, X_test, y_test, n_repeats=5, random_state=42, n_jobs=-1)
importance_df = pd.DataFrame({
    'Feature': safe_features,
    'Importance': result.importances_mean
}).sort_values(by='Importance', ascending=False)

print("=== Top 5 Features (Permutation Importance) ===")
display(importance_df.head(5))
if importance_df['Importance'].iloc[0] > 0.4:
    print("WARNING: Top feature importance is suspiciously high. Check for target leakage.")

# 2. Extract Errors
df_test['is_false_positive'] = ((df_test['rf_prob'] >= 0.5) & (df_test['target_needs_refresh'] == 0))
df_test['is_false_negative'] = ((df_test['rf_prob'] < 0.5) & (df_test['target_needs_refresh'] == 1))

fps = df_test[df_test['is_false_positive']].sort_values(by='rf_prob', ascending=False)
fns = df_test[df_test['is_false_negative']].sort_values(by='rf_prob', ascending=True)

# 3. Print 3 Concrete Wrong Cases
print("\n=== Top 3 False Positives (Model says yes, Reality says no) ===")
display(fps[['content_age_days', 'impressions_90d', 'ctr_decimal', 'rf_prob']].head(3))
print("Why they are hard: These pages likely have high impressions and age (signaling decay to the model), but their actual traffic trend is stable. Lacking time-series trend data, the RF assumes old + high volume = needs refresh.")

print("\n=== Top 3 False Negatives (Model says no, Reality says yes) ===")
display(fns[['content_age_days', 'impressions_90d', 'ctr_decimal', 'rf_prob']].head(3))
print("Why they are hard: These pages are actively decaying, but their raw impression volume might be relatively low or their CTR is unexpectedly high, tricking the model into thinking they are healthy.")

=== Top 5 Features (Permutation Importance) ===


,Feature,Importance
3,impressions_90d,0.223560
5,ctr_decimal,0.011228
11,sessions_90d,0.010320
4,clicks_90d,0.009573
6,avg_position_clean,0.008113



=== Top 3 False Positives (Model says yes, Reality says no) ===


,content_age_days,impressions_90d,ctr_decimal,rf_prob
17602,97,4650,0.0009,0.848028
7665,97,2628,0.0004,0.828531
21077,97,1718,0.0017,0.824238


Why they are hard: These pages likely have high impressions and age (signaling decay to the model), but their actual traffic trend is stable. Lacking time-series trend data, the RF assumes old + high volume = needs refresh.

=== Top 3 False Negatives (Model says no, Reality says yes) ===


,content_age_days,impressions_90d,ctr_decimal,rf_prob
23815,502,665,0.0,0.132348
28032,502,767,0.0,0.184571
586,460,972,0.0,0.192352


Why they are hard: These pages are actively decaying, but their raw impression volume might be relatively low or their CTR is unexpectedly high, tricking the model into thinking they are healthy.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.